# Before Digital: Galvanometers, Bridges, and Magnetic Tape
### *Modern Strain Gage Technology* — Companion Notebook

**Companion notebook to Chapter 25 — Before Digital: Galvanometers, Bridges, and Magnetic Tape.**

Before analog-to-digital converters, the readout device for a Wheatstone bridge was a *galvanometer* — a sensitive current meter whose needle deflected in proportion to the bridge's imbalance current. How small a strain you could resolve came down to three things: the smallest current the galvanometer could detect, the bridge resistance, and the excitation voltage.

The best laboratory galvanometers of the 1950s and 60s could resolve currents on the order of **1 nA**, which in the simple bridge model below translates to roughly **0.04 με** — respectable, but achieved only with careful shielding, vibration isolation, and painstaking zeroing. A modern 24-bit data acquisition system reaches an order of magnitude finer resolution electronically, in milliseconds, without any of those precautions. This notebook works that comparison end to end and lands on the single number that most concisely captures the digital revolution in strain measurement: **~0.003 με** from a standard 24-bit system.

**This notebook demonstrates:**
1. Minimum detectable strain as a function of galvanometer current sensitivity, swept from 1 μA down to 1 nA.
2. A side-by-side plot of that analog-era resolution against the resolution floor of a modern 24-bit ADC.

In [ ]:
# !pip install numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

---
## 1 — Galvanometer resolution and the digital revolution

In the null-balance method of Chapter 25, the galvanometer's smallest detectable current sets the whole measurement's strain resolution. Two short steps connect them. First, modeling the galvanometer as a low-resistance detector across an equal-arm bridge (so it draws the bridge's short-circuit current), the smallest resistance change it can reveal is

$$\Delta R_{\min} = I_{\min}\,\frac{4\,R_g^{2}}{V_{ex}}$$

In words: a smaller detectable current, a lower bridge resistance, or a higher excitation voltage all let the bridge expose a finer resistance change. Second, that resistance change becomes strain through the gage factor,

$$\varepsilon_{\min} = \frac{\Delta R_{\min}}{\mathrm{GF}\cdot R_g}$$

which simply inverts the defining relation $\Delta R / R = \mathrm{GF}\cdot\varepsilon$. The table below sweeps four orders of magnitude in galvanometer sensitivity. Even the best analog-era galvanometer ($I_{\min}\approx 1$ nA) reaches only about **0.04 με** — and only in the laboratory, with the meter shielded from vibration and the bridge patiently balanced. (Because a real galvanometer's coil resistance is comparable to the bridge resistance, its true sensitivity is somewhat worse than this idealized short-circuit limit — a refinement left for the reader below.)

The modern comparison point, **0.003 με**, is not a guess. A standard 24-bit strain module digitizes a full-scale input of about ±25 mV/V; at a gage factor of 2 that corresponds to a full-scale strain of roughly 50,000 με, and dividing it into $2^{24}$ steps leaves about 0.003 με per step. Delta-sigma converters hit this routinely, at kHz sample rates, without any of the galvanometer's precautions. That is the revolution Chapter 25 narrates.

In [ ]:
# --- Analog era: galvanometer current sensitivity -> minimum detectable strain ---
# Equal-arm quarter bridge; the galvanometer is modeled as a low-resistance
# (near short-circuit) detector across the signal diagonal.

R_GAGE = 120.0        # ohm  -- nominal gage / bridge-arm resistance
V_EX = 6.0            # V    -- bridge excitation voltage
GAGE_FACTOR = 2.0     # --   -- strain sensitivity of a typical foil gage

# Galvanometer minimum detectable currents to sweep (A): 1 uA down to 1 nA.
GALVO_SENSITIVITIES = [1e-6, 1e-7, 1e-8, 1e-9]


def min_detectable_strain(i_min, r_gage=R_GAGE, v_ex=V_EX, gage_factor=GAGE_FACTOR):
    """Smallest strain a null-balance bridge can resolve for a given detector current.

    The galvanometer is modeled as a low-resistance detector across an equal-arm
    bridge, so a small arm change dR drives the short-circuit current
    I = V_ex * dR / (4 * R_gage**2). Inverting gives the smallest resolvable dR,
    and dR / R = GF * eps then converts it to strain.

    Parameters
    ----------
    i_min : float or np.ndarray
        Minimum detectable galvanometer current, in amperes.
    r_gage : float
        Gage / bridge-arm resistance, in ohms.
    v_ex : float
        Bridge excitation voltage, in volts.
    gage_factor : float
        Gage factor (dimensionless).

    Returns
    -------
    dR_min : float or np.ndarray
        Minimum detectable resistance change, in ohms.
    eps_min : float or np.ndarray
        Minimum detectable strain (dimensionless; multiply by 1e6 for microstrain).
    """
    dR_min = i_min * 4.0 * r_gage**2 / v_ex
    eps_min = dR_min / (gage_factor * r_gage)
    return dR_min, eps_min


print(f"{'Galvanometer I_min':>18}  {'dR_min (uOhm)':>14}  {'eps_min (ue)':>14}")
print("-" * 50)
for i_min in GALVO_SENSITIVITIES:
    dR_min, eps_min = min_detectable_strain(i_min)
    print(f"{i_min:>15.0e} A  {dR_min * 1e6:>14.4f}  {eps_min * 1e6:>14.4f}")

# --- Modern era: 24-bit ADC resolution floor ---
ADC_BITS = 24
FS_RATIO = 0.025      # V/V -- full-scale input of a standard 24-bit strain module (+/-25 mV/V)

fs_strain = 4.0 * FS_RATIO / GAGE_FACTOR      # full-scale strain (dimensionless) = 0.05 = 50,000 ue
modern_res = fs_strain / 2**ADC_BITS          # strain per ADC step

_, best_galvo_eps = min_detectable_strain(GALVO_SENSITIVITIES[-1])   # best case: 1 nA

print()
print(f"Modern 24-bit ADC: {fs_strain*1e6:,.0f} ue full scale over 2^{ADC_BITS} steps")
print(f"  -> resolution ~{modern_res*1e6:.4f} ue, about "
      f"{best_galvo_eps/modern_res:.0f}x finer than the best 1 nA galvanometer "
      f"(~{best_galvo_eps*1e6:.2f} ue).")

### The picture: analog resolution vs. the digital floor

Numbers in a table are convincing; a picture is unignorable. Plotting minimum detectable strain against galvanometer sensitivity on log–log axes lays the two eras side by side. Each galvanometer point falls on a straight line — resolution scales linearly with detector current — while the modern 24-bit ADC is a flat floor sitting well below even the best analog instrument. The vertical gap between the line and the floor is the digital revolution, drawn to scale.

In [ ]:
i_grid = np.array(GALVO_SENSITIVITIES)
_, eps_grid = min_detectable_strain(i_grid)   # vectorized via NumPy broadcasting

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.loglog(i_grid * 1e9, eps_grid * 1e6, 'o-', color='#c0392b',
          label='Galvanometer null-balance')
ax.axhline(modern_res * 1e6, color='#2c3e50', ls='--',
           label=f'Modern 24-bit ADC (~{modern_res * 1e6:.3f} microstrain)')

ax.set_xlabel(r'Galvanometer minimum detectable current  $I_\mathrm{min}$  (nA)')
ax.set_ylabel(r'Minimum detectable strain  $\varepsilon_\mathrm{min}$  ($\mu\varepsilon$)')
ax.set_title('Strain resolution: analog galvanometer vs. modern 24-bit ADC')
ax.grid(True, which='both', ls=':', alpha=0.5)
ax.legend()
fig.tight_layout()
plt.show()

## Where to take this next

- **Add the galvanometer's own resistance.** The model above assumes an ideal low-resistance detector drawing the bridge's short-circuit current. Real galvanometers had coil resistance comparable to `R_GAGE`, which lowers the signal current. Replace the short-circuit assumption with the full bridge-plus-detector network (Thévenin source resistance `R_GAGE` in series with the coil) and see how much the achievable resolution degrades.
- **Trade excitation against self-heating.** Higher `V_EX` sharpens resolution but heats the gage, with per-arm power `V_EX**2 / (4 * R_GAGE)`. Sweep `V_EX`, plot resolution and gage power on the same axes, and find the practical excitation ceiling where self-heating begins to corrupt the reading.
- **Model the modern noise floor honestly.** A real 24-bit ADC is limited by noise, not just quantization — its *noise-free* resolution is typically several bits short of 24. Fold in an amplifier/ADC noise density (say, an effective 19–20 noise-free bits) and recompute the effective strain resolution for a fairer comparison against the galvanometer.